In [7]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [8]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_juni
Connected to new database: dataleap_v5_migration
Connected to future database: db_future


## Hide code

In [9]:
import pickle
import os

print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 ")
print("================================================================================")

# ================================================================================
# KONFIGURASI 1: LOAD MASING-MASING FILE PICKLE (TETAP TERPISAH)
# ================================================================================
data_cimut = {}
data_afrida = {}
data_hanif = {}

# 1. Load File cimut (Ganti nama file sesuai punyamu)
try:
    with open('fase_3_cimut.pkl', 'rb') as f:
        data_cimut = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik cimut.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_3_afrida.pkl', 'rb') as f:
        data_afrida = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Afrida.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif
try:
    with open('fase_3_hanif.pkl', 'rb') as f:
        data_hanif = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Hanif.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Hanif: {e}")

print("\n================================================================================")
# ================================================================================
# KONFIGURASI 2: ISI DAFTAR TABEL MILIK MASING-MASING ORANG
# ================================================================================
list_table_cimut = [
    "kontak_prospek",
    "calon_siswa",
    "calon_siswa_akademik",
    "calon_siswa_ortu",
    "calon_siswa_bayar",
    "calon_siswa_jadwal",
    "calon_siswa_kursus",
    "calon_siswa_proses",
    "calon_siswa_status_logs",
    "peminjaman",
    "pengadaan",
    "problem",
]

list_table_afrida = [
    "sop",
    "surat_keluar",
    "verifikasi_surat_keluar",
    "surat_tugas",
    "surat_tugas_anggota",
    "sop_kategori",
]

list_table_hanif = [
    "pengajuan_karyawan",
    "histori_pengajuan",
    "pelamar",
    "pelamar_kerja",
    "pelamar_sekolah",
    "pelamar_kursus",
    "progres_pelamar",
    "rekrutmen_pelamar",
]

# ================================================================================
# KONFIGURASI 3: ATUR URUTAN MUTLAK PENYUNTIKAN KE DATABASE (MASTER ORDER)
# ================================================================================
# Masukkan nama tabel yang mau di-insert sesuai urutan FK (Foreign Key).
# Kamu bebas menyilangkan nama tabel di sini, sistem akan otomatis mencari pemiliknya.
master_urutan_insert = [
    "kontak_prospek",
    "calon_siswa",
    "calon_siswa_akademik",
    "calon_siswa_ortu",
    "calon_siswa_bayar",
    "calon_siswa_jadwal",
    "calon_siswa_kursus",
    "calon_siswa_proses",
    "calon_siswa_status_logs",
    "peminjaman",
    "pengadaan",
    "problem",
    "sop",
    "surat_keluar",
    "verifikasi_surat_keluar",
    "surat_tugas",
    "surat_tugas_anggota",
    "sop_kategori",
    "pengajuan_karyawan",
    "histori_pengajuan",
    "pelamar",
    "pelamar_kerja",
    "pelamar_sekolah",
    "pelamar_kursus",
    "progres_pelamar",
    "rekrutmen_pelamar",
]

# ================================================================================
# SISTEM DETEKTIF: MENCARI DAN MENGGABUNGKAN DATA BERDASARKAN PEMILIKNYA
# ================================================================================
print("🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...\n")

data_siap_insert = {}

for table in master_urutan_insert:
    if table in list_table_cimut:
        data_siap_insert[table] = data_cimut.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data cimut.")
        
    elif table in list_table_afrida:
        data_siap_insert[table] = data_afrida.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Afrida.")
        
    elif table in list_table_hanif:
        data_siap_insert[table] = data_hanif.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Hanif.")
        
    else:
        # Jika kamu memasukkan nama tabel di master_urutan tapi lupa memasukkannya di list pemilik
        data_siap_insert[table] = None
        print(f"  ❌ ERROR: Tabel '{table}' tidak ada di list cimut, Afrida, maupun Hanif!")

print("\n✅ Pemetaan selesai! Data siap disuntikkan ke database.")

 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 
✓ Berhasil memuat data PKL milik cimut.
✓ Berhasil memuat data PKL milik Afrida.
✓ Berhasil memuat data PKL milik Hanif.

🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...

  📦 Tabel 'kontak_prospek' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa_akademik' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa_ortu' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa_bayar' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa_jadwal' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa_kursus' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa_proses' otomatis dipetakan dari data cimut.
  📦 Tabel 'calon_siswa_status_logs' otomatis dipetakan dari data cimut.
  📦 Tabel 'peminjaman' otomatis dipetakan dari data cimut.
  📦 Tabel 'pengadaan' otomatis dipetakan dari data cimut.
  📦 Tabel 'problem' otomatis dipetak

## Hide code

In [10]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT (ANTI SILENT-KILLER, AUTO-BATCHING & DIAGNOSTIC)
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list, batch_size=2000):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DENGAN CHUNKING (LOOPING AMAN)
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {'status': 'not_found', 'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl', 'warnings': []}
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {'status': 'empty', 'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)', 'warnings': []}
            continue
            
        try:
                        # Bersihkan kolom kosong murni
            df_to_push = df_target.dropna(axis=1, how='all')
            
            # ================================================================
            # 🔥 KONVERSI FINAL: UBAH SEMUA KOLOM TANGGAL JADI STRING (PASTI BERHASIL!)
            # ================================================================
            for col in df_to_push.columns:
                # Deteksi apakah kolom ini bertipe datetime64 (apapun pecahannya)
                if pd.api.types.is_datetime64_any_dtype(df_to_push[col]):
                    # Ubah menjadi string format MySQL, jika kosong (NaT) jadi None
                    df_to_push[col] = df_to_push[col].apply(
                        lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notnull(x) else None
                    )
            # ================================================================
            
            # Siapkan query
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Langsung convert ke list of tuples (tanpa ribet cleaning isna lagi, karena sudah aman)
            # Tapi kita tetap bersihkan kemungkinan ada NaN/None di kolom lain (angka/string)
            raw_data = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN", ""] else x for x in row) 
                for row in raw_data
            ]
            
            total_rows = len(clean_data_tuples)
            actual_inserted_total = 0
            db_warnings = []
            
            # 🔥 SISTEM AUTO-BATCHING (CHUNKING) 🔥
            # Loop memotong data menjadi bagian-bagian kecil agar MySQL tidak tersedak
            for i in range(0, total_rows, batch_size):
                chunk = clean_data_tuples[i : i + batch_size]
                cursor.executemany(insert_query, chunk)
                
                # Hitung data yang berhasil masuk pada batch ini
                chunk_inserted = max(0, cursor.rowcount)
                actual_inserted_total += chunk_inserted
                
                # Jika ada yang ter-skip di batch ini, tangkap errornya (maksimal simpan 3 per tabel)
                if chunk_inserted < len(chunk) and len(db_warnings) < 3:
                    cursor.execute("SHOW WARNINGS")
                    warnings_fetched = cursor.fetchall()
                    if warnings_fetched:
                        for w in warnings_fetched:
                            w_msg = f"MySQL Warning: {w['Message']}"
                            if w_msg not in db_warnings:
                                db_warnings.append(w_msg)
                            if len(db_warnings) >= 3:
                                break
                                
                # Commit per batch agar memori stabil
                db_connection.commit()
            
            # Evaluasi Status Akhir Tabel
            if actual_inserted_total == total_rows:
                status_flag = 'success'
                msg = f'✓ {table_name}: SEMPURNA! {total_rows}/{total_rows} baris sukses masuk database.'
            else:
                status_flag = 'partial_warning'
                msg = f'⚠️ {table_name}: TER-SKIP! Dikirim {total_rows} baris, tapi yang masuk DB HANYA {actual_inserted_total} baris.'

            results[table_name] = {
                'status': status_flag, 
                'msg': msg,
                'warnings': db_warnings
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'msg': f'✗ {table_name}: Gagal total saat eksekusi insert - Alasan: {e}',
                'warnings': []
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    print("🟢 TABEL YANG 100% SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') == 'success':
            print(f"  {res['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses sempurna)")

    print("\n🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):")
    failed_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') in ['failed', 'partial_warning', 'not_found', 'empty']:
            print(f"  {res['msg']}")
            
            # Cetak alasan dari MySQL (Dibatasi 3 agar tidak merusak tampilan Jupyter)
            if res.get('warnings'):
                for w_msg in res['warnings']:
                    print(f"      -> 🕵️ {w_msg}")
                    
            failed_exist = True
            
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada data yang terbuang.")
            
    print("================================================================================\n")

    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            res = results[table_name]
            
            if res['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name].head(3))
                print("-" * 80)
                
            elif res['status'] in ['failed', 'partial_warning']:
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Pesan Sistem: {res['msg']}")
                print("-" * 50)
                print("Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):")
                display(tables_data[table_name].head(5))
                print(f"\nTipe data Pandas untuk tabel '{table_name}':")
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

## Output

In [11]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL
# ================================================================================
results_fase_3 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=data_siap_insert,       # <--- Menggunakan data yang sudah di-mapping otomatis
    ordered_list=master_urutan_insert   # <--- Menggunakan urutan master buatanmu
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)



 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG 100% SUKSES MASUK:
  ✓ peminjaman: SEMPURNA! 194/194 baris sukses masuk database.
  ✓ pengadaan: SEMPURNA! 110/110 baris sukses masuk database.
  ✓ problem: SEMPURNA! 160/160 baris sukses masuk database.
  ✓ surat_keluar: SEMPURNA! 231/231 baris sukses masuk database.
  ✓ verifikasi_surat_keluar: SEMPURNA! 513/513 baris sukses masuk database.
  ✓ surat_tugas: SEMPURNA! 139/139 baris sukses masuk database.
  ✓ sop_kategori: SEMPURNA! 2/2 baris sukses masuk database.
  ✓ pengajuan_karyawan: SEMPURNA! 33/33 baris sukses masuk database.
  ✓ histori_pengajuan: SEMPURNA! 79/79 baris sukses masuk database.
  ✓ pelamar_kerja: SEMPURNA! 67/67 baris sukses masuk database.

🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):
  ✗ kontak_prospek: Gagal total saat eksekusi insert - Alasan: Failed processing format-parameters; Python 'timestamp' cannot be converted to a MySQL type
  ✗ calon_siswa: Gagal total saat eksekusi inser

,id_kontak_prospek,kode_kontak,nama_penanya,nomor_telepon,email,sumber_informasi,catatan_awal_fo,id_admin_fo,status_kontak,tanggal_kontak_pertama,tanggal_kontak_terakhir
0,1,PIJWREZC,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,Teman/kerabat/saudara,None,None,joined,None,2025-09-18 17:39:40
1,2,LEHUEB6B,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,Instagram,None,None,joined,None,2025-09-23 16:25:14
2,3,HBJ6O8TF,Zulfa Bariatur Rahma,085707179656,chyzryth@gmail.com,Lainnya,None,None,joined,None,2025-10-03 09:48:34
3,4,LI0QW5C1,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,Instagram,None,None,joined,None,2025-10-03 09:49:15
4,5,CEFRJE2H,Khansa Amalia Putri Aji,081554932188,afadhilpa@gmail.com,Teman/kerabat/saudara,None,None,in_disscussion,None,2025-10-03 09:48:50



Tipe data Pandas untuk tabel 'kontak_prospek':
id_kontak_prospek           int64
kode_kontak                object
nama_penanya               object
nomor_telepon              object
email                      object
sumber_informasi           object
catatan_awal_fo            object
id_admin_fo                object
status_kontak              object
tanggal_kontak_pertama     object
tanggal_kontak_terakhir    object
dtype: object
--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: CALON_SISWA] 🚨
Pesan Sistem: ✗ calon_siswa: Gagal total saat eksekusi insert - Alasan: Failed processing format-parameters; Python 'timestamp' cannot be converted to a MySQL type
--------------------------------------------------
Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):


,id_calon,kode_unik,nama_lengkap,id_kontak_prospek,nama_panggilan,jenis_kelamin,tempat_lahir,tanggal_lahir,kewarganegaraan,email,...,fo_status,fo_status_updated_at,handover_at,link_form_sent_at,form_completed_at,first_submitted_at,latest_submitted_at,deleted_at,created_at,updated_at
0,1,PIJWREZC,Daria Azmiya Jasmine,1,Daria,Perempuan,None,None,Indonesia,puterihapsari.f@gmail.com,...,new_lead,None,None,None,None,None,None,None,None,2025-09-18 17:39:40
1,2,LEHUEB6B,Annisa Zahro Ramadhania,2,Annisa,Perempuan,None,None,Indonesia,dwirohm4@gmail.com,...,new_lead,None,None,None,None,None,None,None,None,2025-09-23 16:25:14
2,3,HBJ6O8TF,Zulfa Bariatur Rahma,3,Zulfa,Perempuan,None,None,Indonesia,chyzryth@gmail.com,...,new_lead,None,None,None,None,None,None,None,None,2025-10-03 09:48:34
3,4,LI0QW5C1,Achmad Naufal Albiruni,4,Albi,None,None,None,Indonesia,melisnifuku@gmail.com,...,new_lead,None,None,None,None,None,None,None,None,2025-10-03 09:49:15
4,5,CEFRJE2H,Khansa Amalia Putri Aji,5,Khansa,Perempuan,None,None,Indonesia,afadhilpa@gmail.com,...,new_lead,None,None,None,None,None,None,None,None,2025-10-03 09:48:50


...


Tipe data Pandas untuk tabel 'pelamar_kursus':
id_pelamar_kursus     Int64
id_pelamar            Int64
nama_kursus          object
tanggal              object
deskripsi            object
lokasi               object
nomor_sertifikat     object
dtype: object
--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: PROGRES_PELAMAR] 🚨
Pesan Sistem: ⚠️ progres_pelamar: TER-SKIP! Dikirim 403 baris, tapi yang masuk DB HANYA 304 baris.
--------------------------------------------------
Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):


,id_progres_pelamar,id_pelamar,id_user,status_progres_pelamar,catatan,tautan_file,pertanyaan,created_at
0,11,177,U00001,Interview,,https://drive.google.com/drive/folders/17AhJjH...,-,2023-05-29 16:56:05
1,12,177,U00012,Interview,<p>testing</p>,-,-,2023-05-29 16:57:22
2,13,177,U00001,Interview,<p>aku coba</p>,-,-,2023-05-29 16:59:34
3,14,178,U00001,Tahap Test,<p>interview</p>,https://drive.google.com/drive/folders/1WYB9iR...,-,2023-05-29 17:45:09
4,15,178,U00001,Interview,,,-,2023-05-30 06:21:30



Tipe data Pandas untuk tabel 'progres_pelamar':
id_progres_pelamar         Int64
id_pelamar                 Int64
id_user                   object
status_progres_pelamar    object
catatan                   object
tautan_file               object
pertanyaan                object
created_at                object
dtype: object
--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: REKRUTMEN_PELAMAR] 🚨
Pesan Sistem: ⚠️ rekrutmen_pelamar: TER-SKIP! Dikirim 281 baris, tapi yang masuk DB HANYA 215 baris.
--------------------------------------------------
Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):


,id_rekrutmen,id_pelamar,id_user
0,10,5,U00014
1,12,2,U00014
2,13,2,U00023
3,20,12,U00014
4,21,12,U00016



Tipe data Pandas untuk tabel 'rekrutmen_pelamar':
id_rekrutmen     Int64
id_pelamar       Int64
id_user         object
dtype: object
--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [12]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 1 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_3 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )